# CSA GPU Benchmark - Google Colab

Test Compressed Speculative Attention on GPU

**Requirements:** GPU runtime (T4 or better)

This notebook benchmarks CSA vs baseline generation.

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = "cuda"
else:
    print("WARNING: No GPU - switch to GPU runtime!")
    device = "cpu"

print(f"Device: {device}")

In [ ]:
# Install dependencies and clone CSA
!pip install -q transformers accelerate
!git clone https://github.com/kishoretvk/DevClaw.git
import sys
sys.path.insert(0, '/content/DevClaw')

In [ ]:
# Setup
from csa import CSAEngine
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

CONFIG = {
    'model_name': 'gpt2',
    'max_new_tokens': 50,
    'temperature': 0.7,
    'compression_ratio': 50,
    'compression_frequency': 'once',
    'skip_compression_threshold': 512
}

# Create long prompt (>512 tokens to trigger compression)
prompt = ' AI will transform'.join([str(i) for i in range(200)])

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
prompt_tokens = len(tokenizer.encode(prompt))
print(f"Prompt length: {prompt_tokens} tokens")
print(f"Threshold: {CONFIG['skip_compression_threshold']} (need > this)")

In [ ]:
# Baseline benchmark
print("Running baseline...")
model = AutoModelForCausalLM.from_pretrained(CONFIG['model_name']).to(device)
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
input_ids = tokenizer(prompt, return_tensors='pt').to(model.device)

start = time.time()
with torch.no_grad():
    output = model.generate(
        input_ids.input_ids,
        max_new_tokens=CONFIG['max_new_tokens'],
        do_sample=True,
        temperature=CONFIG['temperature']
    )
baseline_time = time.time() - start
print(f"Baseline time: {baseline_time:.2f}s")

In [ ]:
# Clear GPU memory
import gc
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

In [ ]:
# CSA benchmark
print("Running CSA...")
engine = CSAEngine(
    CONFIG['model_name'],
    compression_ratio=CONFIG['compression_ratio'],
    compression_frequency=CONFIG['compression_frequency'],
    skip_compression_threshold=CONFIG['skip_compression_threshold'],
    use_speculation=False
)

start = time.time()
csa_text = engine.generate(
    prompt,
    max_new_tokens=CONFIG['max_new_tokens'],
    enable_profiling=True
)
csa_time = time.time() - start
print(f"CSA time: {csa_time:.2f}s")

In [ ]:
# Results
print("\n=== RESULTS ===")
print(f"Baseline: {baseline_time:.2f}s")
print(f"CSA: {csa_time:.2f}s")
print(f"Speedup: {baseline_time/csa_time:.2f}x")

if baseline_time < csa_time:
    print("\nNote: CSA is currently slower (adds overhead)")
    print("Compressed cache is computed but not used in generation yet")
else:
    print("\nCSA is faster!")